# Pick and Place with the 2D Two-Link Arm

This notebook drives the `TwoLinkArmWithObject2D` example through a full pick-and-place loop:

1. **Plan** the arm so its tip coincides with the block (block is fixed to the world via a `FixedJoint2D`).
2. **Fire the pickup transition** — the trigger fires when the tip is on the block, swapping the world→block pin for a rigid `link_b`→block attachment.
3. **Plan again** in the new system (block now coupled to the arm) so the block reaches a placement target.
4. **Animate** the concatenated trajectory inline, with markers for the pickup point and the placement target.

In [ ]:
import numpy as np
from IPython.display import HTML
from matplotlib import animation, pyplot
from spatialmath import SE2

from comb.constraints import ConstraintParameters, PointEquality2D
from comb.examples.two_link_arm_with_object_2d import TwoLinkArmWithObject2D
from comb.planners.stepping import SteppingPlanner
from comb.rendering.matplotlib_2d import MatplotlibRenderer2D
from comb.rendering.overlays import PointMarker2D
from comb.trajectories import concatenate

## 1. Build the scene

`TwoLinkArmWithObject2D` packages: a two-link arm, an anchored world body, a `block` body initially pinned to the world via a `FixedJoint2D`, and a `pickup_transition` that fires when the arm's tip coincides with the block.

In [ ]:
ex = TwoLinkArmWithObject2D(block_pose=SE2(0.4, 1.4, 0.0))
block_pickup_xy = (
    float(ex.system.body_poses[ex.block].t[0]),
    float(ex.system.body_poses[ex.block].t[1]),
)
placement_xy = (-0.6, 1.2)
print(f"Block starts at {block_pickup_xy}, will be placed at {placement_xy}")

## 2. Plan the arm tip to the block

We hand the planner the example's `pickup_trigger` directly as the final constraint: it's the same `PointEquality2D` the transition checks, so the planned trajectory ends at exactly the state where the transition fires.

In [ ]:
planner = SteppingPlanner(interval=0.1)
approach = planner.plan(ex.system, [ex.pickup_trigger], horizon=1.0)
state_at_pickup = approach(approach.duration)
print(
    f"Pickup trigger residual at end of approach: {ex.pickup_transition.trigger_residual(state_at_pickup):.2e}"
)
assert ex.pickup_transition.is_enabled(state_at_pickup)

## 3. Fire the pickup transition

Applying `pickup_transition` returns a new `System` in which the world→block `FixedJoint2D` has been removed and replaced by a rigid attachment from `arm.link_b` to the block, capturing their relative transform at this moment. From here on, moving the arm drags the block along.

In [ ]:
system_holding = ex.pickup_transition.apply(ex.system, state_at_pickup)
removed = ex.world_to_block in system_holding.constraints
added = any(
    c.body1 is ex.arm.link_b and c.body2 is ex.block for c in system_holding.constraints
)
print(f"world→block pin still present? {removed} (should be False)")
print(f"arm→block attachment present? {added} (should be True)")

## 4. Plan the block to its placement target

Now we plan in `system_holding`. The placement goal is a `PointEquality2D` saying "the block's body frame is at the placement world coordinates". Position-only (2 residuals) — fits the 2-DoF arm exactly.

In [ ]:
placement_constraint = PointEquality2D(
    body1=ex.world,
    body2=ex.block,
    fixed_parameters=ConstraintParameters(
        values=np.array([placement_xy[0], placement_xy[1], 0.0, 0.0]),
        names=PointEquality2D.fixed_parameter_names(),
    ),
)
place = planner.plan(system_holding, [placement_constraint], horizon=1.0)
trajectory = concatenate([approach, place])
print(f"Total trajectory duration: {trajectory.duration} s")

## 5. Animate

We sample the concatenated trajectory at fixed intervals, push each `SystemState` into `ex.system` (so the renderer picks up the right poses), and render. Two `PointMarker2D` overlays show the pickup point (where the block starts) and the placement target.

In [ ]:
fig, ax = pyplot.subplots(figsize=(5, 5))
renderer = MatplotlibRenderer2D(ax=ax, xlim=(-2.0, 2.0), ylim=(-1.0, 2.5))

samples = list(trajectory.enumerate(0.05))
overlays = [
    PointMarker2D(
        x=block_pickup_xy[0],
        y=block_pickup_xy[1],
        marker="o",
        color="tab:gray",
        size=200.0,
    ),
    PointMarker2D(
        x=placement_xy[0], y=placement_xy[1], marker="*", color="tab:orange", size=300.0
    ),
]


def draw(frame_idx: int):
    _, state = samples[frame_idx]
    for body in ex.system.bodies:
        ex.system.body_poses[body] = state.body_poses[body]
    for c in ex.system.configuration:
        ex.system.configuration[c] = state.configuration[c]
    renderer.render(ex.system, overlays=overlays)
    return []


anim = animation.FuncAnimation(fig, draw, frames=len(samples), interval=50)
pyplot.close(fig)
HTML(anim.to_jshtml())

## What to try next

- Move `placement_xy` somewhere harder to reach (still inside `2 * link_length`) and replay.
- Add a *place* transition: `ConstraintTransition(trigger=placement_constraint, tolerance=..., remove=(<the arm→block joint from system_holding>,))` to detach the block at the end. The third trajectory back to a rest pose would then move the arm without the block.
- Drop the trigger tolerance (`pickup_tolerance` in the example's constructor) and see how close the planner has to get before the transition will fire.